In [5]:
import gradio as gr
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from skimage import util

# --- Helper: Histogram Plot ---
def plot_histogram(img):
    if img is None: return None
    plt.figure(figsize=(3, 2))
    plt.hist(img.flatten(), bins=64, color='gray', alpha=0.7)
    plt.title("Intensity Distribution", fontsize=8)
    plt.xticks(fontsize=6); plt.yticks(fontsize=6)
    plt.tight_layout(); plt.savefig("hist_temp.png"); plt.close()
    return "hist_temp.png"

# --- TAB 1: Restoration & Enhancement ---
def dynamic_restoration(src, n_type, n_lvl, p_method, gamma, f_type, k_size, d0):
    if src is None: return [None]*4 + ["Waiting for Image..."]
    src = cv2.resize(src, (350, 350))
    gray_orig = cv2.cvtColor(src, cv2.COLOR_RGB2GRAY) if len(src.shape) == 3 else src

    # 1. Optional Degradation (Noise Injection)
    noisy = gray_orig.copy()
    if n_type == "Salt & Pepper":
        noisy = util.random_noise(gray_orig/255.0, mode="s&p", amount=n_lvl)
        noisy = (noisy * 255).astype(np.uint8)
    elif n_type == "Gaussian":
        rng = np.random.default_rng(0)
        noise = rng.normal(0, n_lvl, gray_orig.shape).astype(np.float32)
        noisy = np.clip(gray_orig/255.0 + noise, 0, 1)
        noisy = (noisy * 255).astype(np.uint8)
    elif n_type == "Periodic":
        rows, cols = noisy.shape
        x = np.arange(cols)
        noise = 0.10 * np.sin(2 * np.pi * x / 20)
        noise2d = np.tile(noise, (rows, 1)).astype(np.float32)
        img_normalized = noisy / 255.0
        img_p = np.clip(img_normalized + noise2d, 0, 1)
        noisy = (img_p * 255).astype(np.float32)
    elif n_type == "Uniform":
        noise = np.random.uniform(-n_lvl, n_lvl, noisy.shape).astype(np.float32)
        noisy = np.clip(noisy/255.0 + noise, 0, 1)
        noisy = (noisy * 255).astype(np.uint8)

 # 2. Intensity Transformations (Enhancement)
    work = noisy.astype(np.float32)
    if p_method == "Negative":
        work = 255 - work
    elif p_method == "Log":
        c = 255 / np.log(1 + np.max(work))
        work = c * np.log(1 + work)
    elif p_method == "Gamma":
        work = 255 * ((work / 255) ** gamma)
    elif p_method == "Histogram Equalization":
        work = cv2.equalizeHist(work.astype(np.uint8)).astype(np.float32)

    processed = np.clip(work, 0, 255).astype(np.uint8)

    # 3. Filtering (Restoration)
    restored = processed.copy()
    if f_type != "None":
        k = int(k_size) | 1
        rows, cols = processed.shape

        if f_type == "Mean":
            restored = cv2.blur(processed, (k, k))
        elif f_type == "Median":
            from scipy.ndimage import median_filter
            restored = median_filter(processed, size=k)
        elif f_type == "Laplacian":
            lap = cv2.Laplacian(processed, cv2.CV_64F)
            restored = cv2.convertScaleAbs(processed.astype(np.float64) - lap)
        elif f_type in ["Low Pass", "High Pass"]:
            rows, cols = processed.shape
            Fp = np.fft.fftshift(np.fft.fft2(processed))
            crow, ccol = rows//2, cols//2
            U, V = np.ogrid[:rows, :cols]
            D = np.sqrt((U - crow)**2 + (V - ccol)**2)
            mask_f = (D <= d0).astype(np.float32)
            if f_type == "High Pass":
               mask_f = 1 - mask_f
            restored = np.abs(np.fft.ifft2(np.fft.ifftshift(Fp * mask_f))).astype(np.uint8)
        elif f_type == "Notch":
             mask_f = np.ones((rows, cols), np.float32)

             crow, ccol = rows // 2, cols // 2
             notch_radius = max(5, int(d0 * 0.1))

             notch_points = [
                  (crow - int(d0 * 0.5), ccol),
                  (crow + int(d0 * 0.5), ccol),
                  (crow, ccol - int(d0 * 0.5)),
                  (crow, ccol + int(d0 * 0.5)),
             ]

             Y, X = np.ogrid[:rows, :cols]

             for (nr, nc) in notch_points:
                 D = np.sqrt((Y - nr)**2 + (X - nc)**2)
                 mask_f[D <= notch_radius] = 0

    # 4. Quantitative Metrics
    mse_val = np.mean((gray_orig.astype(np.float32) - restored.astype(np.float32))**2)
    psnr = 100 if mse_val == 0 else 20 * np.log10(255.0 / np.sqrt(mse_val))

    return restored, cv2.absdiff(gray_orig, restored), plot_histogram(restored), f"MSE: {mse_val:.2f} | PSNR: {psnr:.2f} dB"


# --- TAB 2: Segmentation & Morphology ---
def dynamic_segmentation(img, seg_meth, threshold, morph_op, se_shape, se_size, class_label):
    if img is None: return [None]*3
    img = cv2.resize(img, (350, 350))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img

    # 1. Segmentation
    if seg_meth == "Global":
        _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    else:
        gray_uint8 = gray.astype(np.uint8)
        binary = cv2.adaptiveThreshold(gray_uint8, 255,
                     cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 8)

    # 2. Morphology
    s_size = int(se_size)
    if se_shape == "Cross":
        se = cv2.getStructuringElement(cv2.MORPH_CROSS, (s_size, s_size))
    elif se_shape == "Disk":
        se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (s_size, s_size))
    else:
        se = np.ones((s_size, s_size), np.uint8)

    m_out = binary.copy()
    if morph_op == "Erosion":
        m_out = cv2.erode(binary, se)
    elif morph_op == "Dilation":
        m_out = cv2.dilate(binary, se)
    elif morph_op == "Opening":
        m_out = cv2.morphologyEx(binary, cv2.MORPH_OPEN, se)
    elif morph_op == "Closing":
        m_out = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, se)
    elif morph_op == "Boundary Extraction":
        eroded = cv2.erode(binary, se, iterations=1)
        m_out = cv2.subtract(binary, eroded)

   # 3. Feature Extraction — FIX: invert mask so black object is detected correctly
    contours, _ = cv2.findContours(m_out, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    contours = sorted(contours, key=cv2.contourArea, reverse=True)
   # Sort contours largest to smallest
    feats = []
    for c in contours:
        area = cv2.contourArea(c)
        if area < 50:
            continue
        perim = cv2.arcLength(c, True)
        circ = (4 * np.pi * area / (perim ** 2)) if perim > 0 else 0

        x, y, w, h = cv2.boundingRect(c)
        aspect_ratio = float(w) / h if h > 0 else 0

        # 1. Compactness (perimeter² / area)
        compactness = round((perim ** 2) / area, 3) if area > 0 else 0

        # 2. Hu Moment M1 (log-scaled first Hu moment)
        moments = cv2.moments(c)
        hu = cv2.HuMoments(moments).flatten()
        hu_m1 = round(-np.sign(hu[0]) * np.log10(np.abs(hu[0]) + 1e-10), 5)

        if moments['m00'] != 0:
            cx = int(moments['m10'] / moments['m00'])
            cy = int(moments['m01'] / moments['m00'])
        else:
            cx, cy = x + w//2, y + h//2
        pts = c[:, 0, :]
        dists = np.sqrt((pts[:, 0] - cx)**2 + (pts[:, 1] - cy)**2)
        sig_mean = round(float(np.mean(dists)), 3)
      # FIX: AspectRatio now saved correctly
        feats.append({
            "Area": round(area, 1),
            "Perimeter": round(perim, 1),
            "Circularity": round(circ, 3),
            "AspectRatio": round(aspect_ratio, 3),
            "Compactness": compactness,
            "Hu_M1": hu_m1,
            "Signature_Mean": sig_mean,
            "Class": class_label
        })

    df = pd.DataFrame(feats)
    df.to_csv("current_sample.csv", index=False)
    return m_out, df, "current_sample.csv"


# --- TAB 3: Batch Analysis ---
def run_classification_analysis(train_file, test_file):
    try:
        df_tr = pd.read_csv(train_file.name) if train_file.name.endswith('.csv') else pd.read_excel(train_file.name)
        df_ts = pd.read_csv(test_file.name) if test_file.name.endswith('.csv') else pd.read_excel(test_file.name)

        cols = ['Area', 'Perimeter', 'Circularity', 'AspectRatio', 'Compactness', 'Hu_M1', 'Signature_Mean']

        # Keep only columns that exist in both files
        cols = [c for c in cols if c in df_tr.columns and c in df_ts.columns]

        scaler = StandardScaler()
        X_all = scaler.fit_transform(np.vstack((df_tr[cols].values, df_ts[cols].values)))
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_all)

        # --- Improved PCA Plot: show class labels on each point ---
        fig, ax = plt.subplots(figsize=(8, 6))

        classes = df_tr['Class'].unique()
        color_map = plt.cm.tab10(np.linspace(0, 1, len(classes)))

        for cls, color in zip(classes, color_map):
            idx = (df_tr['Class'] == cls).values
            ax.scatter(X_pca[:len(df_tr)][idx, 0],
                       X_pca[:len(df_tr)][idx, 1],
                       color=color, label=f'Train: {cls}',
                       s=100, alpha=0.8, edgecolors='black', linewidths=0.5)
            for xi, yi in zip(X_pca[:len(df_tr)][idx, 0], X_pca[:len(df_tr)][idx, 1]):
                ax.annotate(cls, (xi, yi), fontsize=7, color='black',
                            xytext=(4, 4), textcoords='offset points')

        # Test samples
        ax.scatter(X_pca[len(df_tr):, 0], X_pca[len(df_tr):, 1],
                   c='red', marker='x', s=120, linewidths=2,
                   label='Test Samples', zorder=5)
        for i, (xi, yi) in enumerate(zip(X_pca[len(df_tr):, 0], X_pca[len(df_tr):, 1])):
            ax.annotate(f'Test{i+1}', (xi, yi), fontsize=7, color='red',
                        xytext=(4, 4), textcoords='offset points')

        ax.set_title("PCA Cluster Visualization", fontsize=13)
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig("pca_plot.png", dpi=150)
        plt.close()

        # KNN Classification
        preds = []
        for vec in df_ts[cols].values:
            dists = [np.linalg.norm(vec - tr_vec) for tr_vec in df_tr[cols].values]
            preds.append(df_tr.iloc[np.argmin(dists)]['Class'] if 'Class' in df_tr.columns else "Label Missing")
        df_ts['Prediction'] = preds

        return df_ts, "pca_plot.png", f"Success: {len(df_ts)} samples predicted. Features used: {cols}"

    except Exception as e:
        return None, None, f"Error: {str(e)}"


# --- UI Layout ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# DIAPF: Dynamic Image Analysis & Processing Framework")
    with gr.Tabs():

        with gr.TabItem("1. Restoration & Enhancement"):
            with gr.Row():
                with gr.Column(scale=2):
                    src_img = gr.Image(label="Source Image", type="numpy", height=350, width=350)
                    mse_out = gr.Textbox(label="Metrics (MSE & PSNR)")
                with gr.Column(scale=3, variant="panel"):
                    gr.Markdown("### Phase 1 Control")
                    with gr.Accordion("Noise Injection", open=False):
                        n_t = gr.Dropdown(["None", "Salt & Pepper", "Gaussian", "Periodic", "Uniform"],
                                          label="Noise Type", value="None")
                        n_l = gr.Slider(0, 0.2, 0.05, label="Intensity")
                    with gr.Accordion("Filters & Enhancement", open=True):
                        p_m = gr.Radio(["None", "Negative", "Log", "Gamma", "Histogram Equalization"],
                                       label="Transformation", value="None")
                        gam = gr.Slider(0.1, 5.0, 1.0, label="Gamma")
                        f_t = gr.Dropdown(["None", "Mean", "Median", "Laplacian", "Low Pass", "High Pass", "Notch"],
                                          label="Filter", value="None")
                        k_s = gr.Slider(1, 15, 3, step=2, label="Kernel Size")
                        d_0 = gr.Slider(1, 200, 50, label="D0 Cutoff")
                with gr.Column(scale=2):
                    res_out = gr.Image(label="Restored", height=350, width=350)
                    dif_out = gr.Image(label="Diff Map", height=200, width=350)
                    his_out = gr.Image(label="Histogram", height=150, width=350)
            t1_in = [src_img, n_t, n_l, p_m, gam, f_t, k_s, d_0]
            [i.change(dynamic_restoration, t1_in, [res_out, dif_out, his_out, mse_out]) for i in t1_in]

        with gr.TabItem("2. Segmentation & Morphology"):
            with gr.Row():
                with gr.Column(scale=1):
                    seg_src = gr.Image(label="Input Image", height=350, width=350)
                    gr.Markdown("### Phase 2: Structural Extraction")
                    sm = gr.Radio(["Global", "Adaptive"], label="Method", value="Global")
                    st = gr.Slider(0, 255, 127, label="Threshold")
                    mo = gr.Dropdown(["None", "Erosion", "Dilation", "Opening", "Closing", "Boundary Extraction"],
                                     label="Operation", value="None")
                    sh = gr.Radio(["Square", "Cross", "Disk"], label="Shape", value="Square")
                    sz = gr.Slider(3, 15, 3, step=2, label="Size")
                with gr.Column(scale=1):
                    bin_out = gr.Image(label="Binary Mask", height=350, width=350)
                    feat_out = gr.DataFrame(label="Features Vector", interactive=True)
                    gr.Markdown("> **NOTE:** Features now include: Area, Perimeter, Circularity, AspectRatio, **Compactness, Hu_M1, Signature_Mean**")
                    c_lbl = gr.Textbox(label="Class Name", placeholder="e.g. Car or Truck", interactive=True)
                    file_out = gr.File(label="Download CSV Record")
            t2_in = [seg_src, sm, st, mo, sh, sz, c_lbl]
            [i.change(dynamic_segmentation, t2_in, [bin_out, feat_out, file_out]) for i in t2_in]

        with gr.TabItem("3. Classification & PCA"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Phase 3: Analytical Analysis")
                    gr.Markdown("Upload merged CSV files. Training must have **Class** column. Test samples leave Class blank.")
                    ftr = gr.File(label="Upload Training Reference (CSV)")
                    fts = gr.File(label="Upload Final Test Samples (CSV)")
                    bc = gr.Button("Run Analysis", variant="primary")
                with gr.Column():
                    pp = gr.Image(label="PCA Mapping")
                    cr = gr.Textbox(label="Status Report")
            ct = gr.DataFrame(label="Final Results Table")
            bc.click(run_classification_analysis, [ftr, fts], [ct, pp, cr])

demo.launch()

/tmp/ipykernel_1391/356061216.py:261: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b57db1429de876ceb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
